## Scenario 2: A cross-functional team with one data scientist working on an ML model


MLflow setup:
- tracking server: yes, local server
- backend store: sqlite database
- artifacts store: local filesystem

The experiments can be explored locally by accessing the local tracking server.

To run this example you need to launch the mlflow server locally by running the following command in your terminal:

`mlflow server --backend-store-uri sqlite:///backend.db`

In [1]:
import mlflow


mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5000'


In [3]:
mlflow.search_experiments()

[<Experiment: artifact_location='/workspaces/zoomcamp_MLops_launch/mlruns/2', creation_time=1789141383022, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1789141383022, lifecycle_stage='active', name='Model-Registry', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/zoomcamp_MLops_launch/mlruns/1', creation_time=1788126385169, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788126385169, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1788111050205, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1788111050205, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("MLflow in practice session")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="LogisticReg_models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/09/21 10:11:08 INFO mlflow.tracking.fluent: Experiment with name 'MLflow in practice session' does not exist. Creating a new experiment.
2026/09/21 10:11:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


default artifacts URI: 'mlflow-artifacts:/3/f3e9ba87a1dc4cbcaa3f5a08019388e6/artifacts'
🏃 View run youthful-panda-936 at: http://127.0.0.1:5000/#/experiments/3/runs/f3e9ba87a1dc4cbcaa3f5a08019388e6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1789985468086, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1789985468086, lifecycle_stage='active', name='MLflow in practice session', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/zoomcamp_MLops_launch/mlruns/2', creation_time=1789141383022, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1789141383022, lifecycle_stage='active', name='Model-Registry', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/zoomcamp_MLops_launch/mlruns/1', creation_time=1788126385169, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788126385169, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1788111050205, effective_trace_archiv

### Interacting with the model registry

In [6]:
from mlflow.tracking import MlflowClient


client = MlflowClient("http://127.0.0.1:5000")

In [7]:
client.search_registered_models()

[<RegisteredModel: aliases={'staging': '2'}, creation_timestamp=1789141760008, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description='best xgboost models for predicting taxi duration ', last_updated_timestamp=1789141962692, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1789141848919, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1789141848919, metrics=None, model_id=None, name='best_xgboost', params=None, run_id='923db63eab5843f294ed1e2d604fcc44', run_link='', source='models:/m-bad3d82783884ec5af354f44b469fbe5', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id='', version='2', workspace='default'>], name='best_xgboost', tags={'model': 'xgboost'}, workspace='default'>,
 <RegisteredMo

In [8]:
run_id = client.search_runs(experiment_ids='3')[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/LogisticReg_models",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2026/09/21 10:13:49 WARNING mlflow.tracking._model_registry.fluent: Run with id f3e9ba87a1dc4cbcaa3f5a08019388e6 has no artifacts at artifact path 'LogisticReg_models', registering model based on models:/m-54961c3cc7c84616bef90134eefa65da instead
2026/09/21 10:13:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1789985629700, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1789985629700, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='f3e9ba87a1dc4cbcaa3f5a08019388e6', run_link='', source='models:/m-54961c3cc7c84616bef90134eefa65da', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>